# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [21]:
import os, subprocess
import pandas as pd
import numpy as np

# Clone repo if needed
REPO_URL = "https://github.com/hibathakur559-boop/flyrank-ml-Hiba"
REPO_DIR = "flyrank-ml-Hiba"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")

# ============================================================
# SIGNAL CHECK 1: staleness (days_since_last_update) vs decline
# This is the signal behind FlyRank's "stale_visible_page" refresh flag.
# ============================================================
stale_check = df[df['impressions_90d'] >= 100].copy()
stale_check['staleness_bucket'] = pd.cut(
    stale_check['content_age_days'],
    bins=[0, 90, 180, 365, 10000],
    labels=['0-90d', '90-180d', '180-365d', '365d+']
)
bucket1 = stale_check.groupby('staleness_bucket', observed=True).agg(
    n=('trend_direction', 'size'),
    pct_declining=('trend_direction', lambda x: (x == 'down').mean())
)
print("\nSignal 1: content age vs declining trend")
print(bucket1)

# Verdict: does older content decline more?
oldest_decline = bucket1.loc['365d+', 'pct_declining']
newest_decline = bucket1.loc['0-90d', 'pct_declining']
print(f"\nVerdict basis: 365d+ decline rate = {oldest_decline:.2%}, "
      f"0-90d decline rate = {newest_decline:.2%}")
# CONFIRMED if older content declines noticeably more often

# ============================================================
# SIGNAL CHECK 2: CTR vs position tier
# This is the signal behind FlyRank's "low_ctr_visible_page" / CTR-fix logic.
# ============================================================
ctr_check = df[df['impressions_90d'] >= 100].copy()
ctr_check['position_bucket'] = pd.cut(
    ctr_check['avg_position'],
    bins=[0, 3, 10, 20, 1000],
    labels=['1-3', '4-10', '11-20', '20+']
)
bucket2 = ctr_check.groupby('position_bucket', observed=True).agg(
    n=('ctr', 'size'),
    avg_ctr=('ctr', 'mean')
)
print("\nSignal 2: position tier vs average CTR")
print(bucket2)
# CONFIRMED if CTR clearly drops as position tier gets worse

Loaded 30000 rows

Signal 1: content age vs declining trend
                     n  pct_declining
staleness_bucket                     
0-90d              304       0.680921
90-180d           8717       0.691752
180-365d          7650       0.606013
365d+             5335       0.427179

Verdict basis: 365d+ decline rate = 42.72%, 0-90d decline rate = 68.09%

Signal 2: position tier vs average CTR
                    n   avg_ctr
position_bucket                
1-3               555  0.337153
4-10             8660  0.354024
11-20            5876  0.255689
20+              6915  0.131067


In [22]:
# ============================================================
# VERDICTS
# ============================================================
# Signal 1 (staleness vs decline): OPPOSITE
#   Older content (365d+) declines LESS (42.7%) than newer content (68.1%).
#   This contradicts the common assumption that old = declining. I will NOT
#   use raw content_age as a decline signal in my rule - this negative
#   result just saved my rule from a false assumption.
#
# Signal 2 (CTR vs position tier): CONFIRMED
#   CTR clearly drops as position tier worsens (33.7% at position 1-3 down
#   to 13.1% at position 20+). This is a real, usable signal - it is also
#   the signal behind FlyRank's low_ctr_visible_page / CTR-fix flag.

# ============================================================
# THE RULE: CTR gap below position-tier expectation, weighted by demand
# ============================================================
expected_ctr_by_tier = ctr_check.groupby('position_bucket', observed=True)['ctr'].mean()

df2 = df[df['impressions_90d'] >= 100].copy()
df2['position_bucket'] = pd.cut(
    df2['avg_position'], bins=[0, 3, 10, 20, 1000],
    labels=['1-3', '4-10', '11-20', '20+']
)

# Map expected CTR by tier, then force both columns to plain float
# (avoids the categorical dtype subtraction error)
df2['expected_ctr'] = df2['position_bucket'].map(expected_ctr_by_tier).astype(float)
df2['ctr'] = df2['ctr'].astype(float)
df2['ctr_gap'] = df2['expected_ctr'] - df2['ctr']  # positive = underperforming

# Score: how far below expected CTR, weighted by how much demand exists
df2['score'] = df2['ctr_gap'].clip(lower=0) * np.log1p(df2['impressions_90d'])

df2['reason_code'] = 'ctr_below_tier_expectation'
df2['action'] = 'review_title_and_meta'

print("Rule: score = (expected_ctr_for_tier - actual_ctr) * log(1 + impressions)")
print("Reason code: ctr_below_tier_expectation")
print("Action: review_title_and_meta")
print(f"\nPages scored: {len(df2)}")

Rule: score = (expected_ctr_for_tier - actual_ctr) * log(1 + impressions)
Reason code: ctr_below_tier_expectation
Action: review_title_and_meta

Pages scored: 22006


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
import os

# Rank the queue by score, descending
ranked_queue = df2.sort_values('score', ascending=False).reset_index(drop=True)
ranked_queue['rank'] = ranked_queue.index + 1

# Select the columns a reviewer actually needs
output_cols = ['rank', 'content_id', 'client_id', 'position_bucket',
                'avg_position', 'ctr', 'expected_ctr', 'ctr_gap',
                'impressions_90d', 'score', 'reason_code', 'action']
final_queue = ranked_queue[output_cols]

# Write to the required path
os.makedirs("work/outputs", exist_ok=True)
final_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

n_rows = len(final_queue)
msg = "Wrote " + str(n_rows) + " ranked rows to work/outputs/baseline_action_score.csv"
print(msg)
final_queue.head(10)

Wrote 22006 ranked rows to work/outputs/baseline_action_score.csv


,rank,content_id,client_id,position_bucket,avg_position,ctr,expected_ctr,ctr_gap,impressions_90d,score,reason_code,action
0,1,content_c8e9d6ab9013,client_19581e27de,4-10,9.7,0.00,0.354024,0.354024,208678,4.336285,ctr_below_tier_expectation,review_title_and_meta
1,2,content_453722754fea,client_f369cb89fc,4-10,7.6,0.01,0.354024,0.344024,140079,4.076677,ctr_below_tier_expectation,review_title_and_meta
2,3,content_39881853ef0c,client_f369cb89fc,4-10,7.2,0.01,0.354024,0.344024,112434,4.001047,ctr_below_tier_expectation,review_title_and_meta
3,4,content_c84a0ab98e90,client_f369cb89fc,4-10,7.8,0.03,0.354024,0.324024,223271,3.990730,ctr_below_tier_expectation,review_title_and_meta
4,5,content_0919dd345d80,client_4e07408562,4-10,7.0,0.02,0.354024,0.334024,119217,3.904312,ctr_below_tier_expectation,review_title_and_meta
5,6,content_4a6607efcb46,client_6208ef0f77,1-3,2.2,0.01,0.337153,0.327153,128068,3.847427,ctr_below_tier_expectation,review_title_and_meta
6,7,content_8451fc6f034d,client_d029fa3a95,1-3,2.3,0.03,0.337153,0.307153,272144,3.843742,ctr_below_tier_expectation,review_title_and_meta
7,8,content_36ff89c8214e,client_19581e27de,4-10,7.3,0.05,0.354024,0.304024,295097,3.829205,ctr_below_tier_expectation,review_title_and_meta
8,9,content_c1fe78bc4e37,client_19581e27de,4-10,7.5,0.03,0.354024,0.324024,134055,3.825434,ctr_below_tier_expectation,review_title_and_meta
9,10,content_d274ac4158ef,client_4e07408562,4-10,6.8,0.01,0.354024,0.344024,65138,3.813261,ctr_below_tier_expectation,review_title_and_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [24]:
# Top-20 review: for each, one line on action, why, and what would make it wrong

top20 = final_queue.head(20).copy()
print(top20[['rank', 'content_id', 'position_bucket', 'ctr', 'expected_ctr',
             'impressions_90d', 'action']].to_string(index=False))

print()
print("Manual review notes:")
print()
print("Rank 1-10 pattern: all show near-zero CTR (0.00-0.05) against a 33-35%")
print("tier expectation, with very high impressions (65k-295k) - meaning many")
print("people SEE these pages in search results but almost nobody clicks.")
print("This is exactly the visible-but-under-clicked pattern the rule targets.")
print()
print("Action for all: review_title_and_meta - the title/snippet likely does")
print("not match what searchers expect, or looks unappealing next to competitors.")
print()
print("What would make each one WRONG:")
print("1. If low CTR is caused by a SERP feature (featured snippet, AI")
print("   overview) stealing the click before the user reaches the listing -")
print("   a title rewrite would not fix that, only monitoring would help.")
print("2. If the page recently changed URL or redirected and CTR has not")
print("   stabilized yet - the problem may resolve on its own without an edit.")
print("3. If impressions come from a mismatched query (page ranks for the")
print("   wrong intent) - rewriting the title fixes a symptom, not the cause;")
print("   the real fix would be content and topic alignment.")
print("4. If one client dominates the top ranks - worth checking this is not")
print("   a single client's broad issue skewing the whole ranked queue.")

 rank           content_id position_bucket  ctr  expected_ctr  impressions_90d                action
    1 content_c8e9d6ab9013            4-10 0.00      0.354024           208678 review_title_and_meta
    2 content_453722754fea            4-10 0.01      0.354024           140079 review_title_and_meta
    3 content_39881853ef0c            4-10 0.01      0.354024           112434 review_title_and_meta
    4 content_c84a0ab98e90            4-10 0.03      0.354024           223271 review_title_and_meta
    5 content_0919dd345d80            4-10 0.02      0.354024           119217 review_title_and_meta
    6 content_4a6607efcb46             1-3 0.01      0.337153           128068 review_title_and_meta
    7 content_8451fc6f034d             1-3 0.03      0.337153           272144 review_title_and_meta
    8 content_36ff89c8214e            4-10 0.05      0.354024           295097 review_title_and_meta
    9 content_c1fe78bc4e37            4-10 0.03      0.354024           134055 review_title

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [25]:
# Weak picks + leakage check

print("Weak picks (lower-confidence rows in the queue):")
print()
weak_picks = final_queue[(final_queue['impressions_90d'] < 200) &
                           (final_queue['score'] > 0)].sort_values('score', ascending=False).head(5)
print(weak_picks[['rank', 'content_id', 'ctr', 'expected_ctr', 'impressions_90d', 'score']].to_string(index=False))
print()
print("These are weak picks because impressions_90d is low (under 200) - the")
print("CTR gap could be noise from a small sample rather than a real problem.")
print("A single extra click on a low-impression page swings CTR a lot, so")
print("these should be reviewed with less confidence than the top-10 rows.")

print()
print("Leakage check:")
print("1. No product decision flags used - I only used impressions_90d,")
print("   avg_position, and ctr, all raw observed signals, not FlyRank's")
print("   own health_score, priority_score, or action_type.")
print("2. No future-window data used - expected_ctr was computed from the")
print("   SAME window as the page being scored (impressions_90d, ctr), not")
print("   from any later time period. This is a single-snapshot baseline,")
print("   not a future-outcome prediction, so there is no train/target leak.")
print("3. reason_code and action were derived only from ctr_gap and score,")
print("   which are themselves derived only from ctr/position/impressions -")
print("   no circular use of a label or a rebuilt product flag anywhere.")

Weak picks (lower-confidence rows in the queue):

 rank           content_id  ctr  expected_ctr  impressions_90d    score
 2827 content_b73061588e7d  0.0      0.354024              199 1.875733
 2828 content_c694763d4685  0.0      0.354024              199 1.875733
 2829 content_cf452780e7bb  0.0      0.354024              199 1.875733
 2837 content_128de35a7595  0.0      0.354024              198 1.873958
 2839 content_958a46db26bd  0.0      0.354024              198 1.873958

These are weak picks because impressions_90d is low (under 200) - the
CTR gap could be noise from a small sample rather than a real problem.
A single extra click on a low-impression page swings CTR a lot, so
these should be reviewed with less confidence than the top-10 rows.

Leakage check:
1. No product decision flags used - I only used impressions_90d,
   avg_position, and ctr, all raw observed signals, not FlyRank's
   own health_score, priority_score, or action_type.
2. No future-window data used - expected_

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.